# Visualize Model Predictions

In [ ]:
import json
import sys
import os
from pathlib import Path
import h5py

import numpy as np
import matplotlib.pyplot as plt
import torch

# Import model architecture.
CWD = os.path.join(Path().resolve())
sys.path.append(os.path.abspath(os.path.join(CWD, '..', 'training')))
from models import build_model

In [ ]:
# Specify job_id and hyperparamter config id.
config_id = "aramis_100ms"
job_id = 48

# Path to model weights and config file.
job_dir = os.path.abspath(os.path.join(
    CWD, "../trained_models/{0:}/{1:05g}".format(config_id, job_id)
))

# Load config file.
with open(os.path.join(job_dir, 'config.json')) as f:
    CONFIG = json.load(f)

# Load model.
if not torch.cuda.is_available():
    CONFIG['DEVICE'] = 'CPU'
    model, _ = build_model(CONFIG)
    model.load_state_dict(
        torch.load(
            os.path.join(job_dir, "best_weights.pt"),
            map_location=torch.device('cpu')))
else:
    model, _ = build_model(CONFIG)
    model.load_state_dict(torch.load(os.path.join(job_dir, 'best_weights.pt')))
model.eval();

# Specify data file.
data_dir = os.path.abspath(os.path.join(CWD, "../data"))
data_file = h5py.File(os.path.join(data_dir, "train_set.h5"))

In [ ]:
# Plot example vocalization
plt.plot(data_file["vocalizations"][np.random.randint(100)].T, alpha=0.3)

In [ ]:
def compute_model_prediction(idx):
    audio_inputs = torch.from_numpy(data_file["vocalizations"][idx][None, :, :])
    with torch.no_grad():
        return model(audio_inputs)[0].numpy()

def plot_model_prediction(idx):
    targ = data_file["locations"][idx]
    x, y = targ[:, :, 0], targ[:, :, 1]
    pred = compute_model_prediction(idx)
    xhat, yhat = pred[:, :, 0], pred[:, :, 1]
    for c, color in enumerate(("r", "g")):
        plt.scatter(x[:, c], y[:, c], edgecolors=color, facecolors='none', s=100)
        plt.scatter(xhat[:, c], yhat[:, c], color=color, marker="+")
        
    plt.xlim([-.5, .5])
    plt.ylim([-.3, .3])
    plt.title(idx)

In [ ]:
from scipy.spatial.distance import pdist
meanlocs = np.array(data_file["locations"]).mean(axis=1)
maxdists = []
for i, M in enumerate(meanlocs):
    maxdists.append(pdist(M).max())
sorted_idx = np.argsort(maxdists)[::-1]

In [ ]:
plot_model_prediction(np.random.randint(7895))

In [ ]:
audio_input = torch.from_numpy(data_file["vocalizations"][10][None, :, :])
with torch.no_grad():
    h = (torch.tanh(model.f_convs[0](audio_input)) * torch.sigmoid(model.g_convs[0](audio_input)))[0]
# with torch.no_grad():
#     h = torch.relu(model.f_convs[0](audio_input))[0]

In [ ]:
plt.hist(h[0][::10].numpy(), np.linspace(-1, 1, 50))

In [ ]:
plt.hist(h[0][::10])

In [ ]:
plt.plot(audio_input[0][0][::10])